# Data Exploration: SoundCam RIRs and MedleyDB Input Signals (Legacy-Style Animations)

This notebook visualises the datasets used in the DDSP-ARE experiments, with room-layout animations aligned to the legacy SoundCam EDA style.

It includes:
1. SoundCam RIR loading and delay inspection.
2. Room layout with SoundCam geometry (mm coordinates).
3. Static and animated response visualisations.
4. Interpolation transition comparison.
5. MedleyDB and compensation target previews.

In [ ]:
import sys
from pathlib import Path

root = Path().resolve().parents[1]
sys.path.insert(0, str(root / 'src'))
sys.path.insert(0, str(root / 'src' / 'external'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import torch
import torchaudio

from utils.common import load_rirs, get_delay_from_ir, interpolate_IRs
from utils.plotting import configure_text_rendering, log_smooth_curve

configure_text_rendering()
%matplotlib inline
print(f'Repository root: {root}')

---
## 1. Load SoundCam RIRs

In [ ]:
rir_dir_ml = root / 'data' / 'SoundCam' / 'moving_listener'
rir_dir_mp = root / 'data' / 'SoundCam' / 'moving_person'

rirs_ml, srs_ml = load_rirs(rir_dir_ml, max_n=100, normalize=False)
rirs_mp, srs_mp = load_rirs(rir_dir_mp, max_n=100, normalize=False)
sr = srs_ml[0]

print(f'Moving listener:  {len(rirs_ml):3d} RIRs  |  sr = {sr} Hz  |  length = {len(rirs_ml[0])} samples ({len(rirs_ml[0])/sr:.2f} s)')
print(f'Moving person:    {len(rirs_mp):3d} RIRs  |  sr = {srs_mp[0]} Hz  |  length = {len(rirs_mp[0])} samples ({len(rirs_mp[0])/sr:.2f} s)')

print('\nDirect-sound delays (moving_listener):')
for i, rir in enumerate(rirs_ml):
    d = get_delay_from_ir(rir, sr)
    print(f'  RIR {i:2d}: delay = {d:5d} samples  ({d/sr*1000:.1f} ms)')

---
## 2. Room Layout and Position Map (SoundCam Geometry)

In [ ]:
n_ml = len(rirs_ml)
n_mp = len(rirs_mp)

length = 258 + 3 / 8
width = 104 + 3 / 16
mic_height = 51.5 + 13 / 16

mic_1 = np.array([98 + 1 / 8, -253.75, mic_height]) * 25.4
mic_2 = np.array([53 + 1 / 16, -255.5, mic_height]) * 25.4
mic_3 = np.array([7 + 7 / 8, -(length - 53), mic_height]) * 25.4
mic_4 = np.array([5 + 13 / 16, -(length - (101 + 7 / 8)), mic_height]) * 25.4
mic_5 = np.array([6 + 7 / 8, -52.5, mic_height]) * 25.4
mic_6 = np.array([14 + 5 / 8, -12.75, mic_height]) * 25.4
mic_7 = np.array([width - 17.25, -10, mic_height]) * 25.4
mic_8 = np.array([width - (7 + 3 / 8), -47.5, mic_height]) * 25.4
mic_9 = np.array([width - (7 + 13 / 16), -161.25, mic_height]) * 25.4
mic_10 = np.array([width - 8, -(224 + 7 / 8), mic_height]) * 25.4
mic_xyzs = np.stack((mic_1, mic_2, mic_3, mic_4, mic_5, mic_6, mic_7, mic_8, mic_9, mic_10), axis=0)

speaker_bottom_left = np.array([3.5 + 57.5, -17, 8 + 1 / 8 + 28 + 13 / 16]) * 25.4
speaker_bottom_right = np.array([3.5 + 57.5 + 8.75, -17, 8 + 1 / 8 + 28 + 13 / 16]) * 25.4
speaker_top_left = np.array([3.5 + 57.5, -17, 8 + 1 / 8 + 28 + 13 / 16 + 17]) * 25.4
speaker_top_right = np.array([3.5 + 57.5 + 8.75, -17, 8 + 1 / 8 + 28 + 13 / 16 + 17]) * 25.4
speaker_xyz = (speaker_bottom_left + speaker_bottom_right + speaker_top_left + speaker_top_right) / 4

top_left = np.array([0, 0]) * 25.4
top_right = np.array([width, 0]) * 25.4
bottom_left = np.array([0, -length]) * 25.4
bottom_right = np.array([width, -length]) * 25.4
walls = np.array([top_left, top_right, bottom_right, bottom_left, top_left])

x_min, x_max = top_left[0], top_right[0]
y_min, y_max = bottom_right[1], top_right[1]

ml_xy = mic_xyzs[:n_ml, :2]
ml_x, ml_y = ml_xy[:, 0], ml_xy[:, 1]

centroid_candidates = [
    root / 'data' / 'SoundCam' / 'moving_person' / 'centroid.npy',
    root / 'data' / 'SoundCam' / 'moving_person_centroid.npy',
    root / 'data' / 'SoundCam' / 'centroid.npy',
]

mp_centroids = None
for cand in centroid_candidates:
    if cand.exists():
        mp_centroids = np.load(cand)
        break

if mp_centroids is not None and len(mp_centroids) >= n_mp:
    mp_x = mp_centroids[:n_mp, 0]
    mp_y = mp_centroids[:n_mp, 1]
    mode = 'centroids'
else:
    n_cols_mp = int(np.ceil(np.sqrt(n_mp)))
    n_rows_mp = int(np.ceil(n_mp / n_cols_mp))
    mp_idx = np.arange(n_mp)
    mp_x = x_min + (0.08 + (mp_idx % n_cols_mp) / max(n_cols_mp - 1, 1) * 0.84) * (x_max - x_min)
    mp_y = y_min + (0.08 + (mp_idx // n_cols_mp) / max(n_rows_mp - 1, 1) * 0.84) * (y_max - y_min)
    mode = 'index_fallback'

speaker_pos = tuple(speaker_xyz[:2])
mic_fixed = tuple(mic_xyzs[1, :2])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, title, xs, ys, label in [
    (axes[0], 'Moving Listener Scenario', ml_x, ml_y, 'Mic position'),
    (axes[1], f'Moving Person Scenario ({mode})', mp_x, mp_y, 'Person position'),
]:
    ax.plot(walls[:, 0], walls[:, 1], 'k-', lw=2, label='Walls')
    n = len(xs)
    colours = plt.get_cmap('coolwarm')(np.linspace(0, 1, n))
    ax.scatter(xs, ys, c=colours, s=50, zorder=3, label=label, edgecolors='black', linewidths=0.3)
    if ax is axes[0]:
        for i, (mx, my) in enumerate(zip(xs, ys)):
            ax.annotate(str(i + 1), (mx, my), fontsize=7, ha='center', va='bottom', xytext=(0, 4), textcoords='offset points')
    ax.plot(*speaker_pos, 'rs', markersize=9, label='Speaker')
    if ax is axes[1]:
        ax.plot(*mic_fixed, 'g^', markersize=8, label='Microphone (fixed)')
    ax.set_xlim(x_min - 200, x_max + 200)
    ax.set_ylim(y_min - 200, y_max + 200)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, linestyle=':', alpha=0.4)

fig.suptitle('SoundCam Room Layout (mm coordinates)')
fig.tight_layout()
plt.show()

---
## 3. Static Magnitude Response Comparison

In [ ]:
NFFT = 2 ** 15
freqs = np.fft.rfftfreq(NFFT, d=1.0 / sr)

def compute_mag_db(rir, nfft=NFFT):
    H = np.fft.rfft(rir, n=nfft)
    return 20.0 * np.log10(np.abs(H) + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, rirs, title in [
    (axes[0], rirs_ml, f'Moving Listener ({len(rirs_ml)} positions)'),
    (axes[1], rirs_mp, f'Moving Person ({len(rirs_mp)} positions)'),
]:
    n = len(rirs)
    colours = plt.get_cmap('coolwarm')(np.linspace(0, 1, n))
    for i, rir in enumerate(rirs):
        mag = compute_mag_db(rir)
        m = freqs > 0
        f_s, mag_s = log_smooth_curve(freqs[m], mag[m], window_pts=201)
        ax.semilogx(f_s, mag_s, color=colours[i], linewidth=0.8, alpha=0.75)
    sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(0, n - 1))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label='Position index')
    ax.set_xlim(50, 20000)
    ax.set_ylim(-30, 15)
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Magnitude [dB]')
    ax.set_title(title)
    ax.grid(True, linestyle=':', alpha=0.6)

fig.suptitle('RIR Magnitude Spectra - all positions')
fig.tight_layout()
plt.show()

---
## 4. Animation - Moving Listener Scenario

In [ ]:
def make_mag_animation(rirs, sr, title, nfft=NFFT, fps=4):
    freqs_local = np.fft.rfftfreq(nfft, d=1.0 / sr)
    m = freqs_local > 50
    all_f, all_mag = [], []
    for rir in rirs:
        mag = compute_mag_db(rir, nfft=nfft)
        f_s, mag_s = log_smooth_curve(freqs_local[m], mag[m], window_pts=201)
        all_f.append(f_s)
        all_mag.append(mag_s)
    y_min = min(np.min(m_) for m_ in all_mag) - 3
    y_max = max(np.max(m_) for m_ in all_mag) + 3
    fig_a, ax_a = plt.subplots(figsize=(8, 4))
    line, = ax_a.semilogx([], [], linewidth=1.5, color='steelblue')
    ax_a.set_xlim(50, 20000)
    ax_a.set_ylim(y_min, y_max)
    ax_a.set_xlabel('Frequency [Hz]')
    ax_a.set_ylabel('Magnitude [dB]')
    ax_a.grid(True, linestyle=':', alpha=0.6)
    title_text = ax_a.set_title('')
    def init():
        line.set_data([], [])
        return (line,)
    def update(frame):
        line.set_data(all_f[frame], all_mag[frame])
        title_text.set_text(f'{title} - Position {frame}')
        return (line, title_text)
    ani = animation.FuncAnimation(fig_a, update, frames=len(rirs), init_func=init, interval=int(1000 / fps), blit=True)
    plt.close(fig_a)
    return HTML(ani.to_jshtml())

make_mag_animation(rirs_ml, sr, 'Moving Listener')

---
## 5. Animation - Moving Person Scenario

In [ ]:
make_mag_animation(rirs_mp, sr, 'Moving Person', fps=10)

---
## 6. Animation - Room Layout with Moving Position

In [ ]:
def make_layout_animation(xs, ys, speaker_pos, mic_fixed_pos, scenario_label, fps=4):
    n = len(xs)
    fig_l, ax_l = plt.subplots(figsize=(5.8, 5.2))
    ax_l.plot(walls[:, 0], walls[:, 1], 'k-', lw=2, label='Walls')
    ax_l.scatter([speaker_pos[0]], [speaker_pos[1]], c='red', s=120, marker='s', label='Speaker', edgecolors='black', linewidths=0.5, zorder=6)
    if mic_fixed_pos is not None:
        ax_l.scatter([mic_fixed_pos[0]], [mic_fixed_pos[1]], c='green', s=90, marker='^', label='Microphone (fixed)', edgecolors='black', linewidths=0.5, zorder=6)
    ax_l.plot(xs, ys, 'b-', alpha=0.25, lw=1, label='Trajectory')
    ax_l.scatter(xs, ys, c='lightgray', s=30, zorder=2)
    active_dot, = ax_l.plot([], [], 'o', color='red', markersize=10, markeredgecolor='black', markeredgewidth=1.2, zorder=10, label='Active position')
    trail_scatter = ax_l.scatter([], [], c=[], cmap='Reds', s=35, alpha=0.7, zorder=9)
    ax_l.set_xlim(x_min - 200, x_max + 200)
    ax_l.set_ylim(y_min - 200, y_max + 200)
    ax_l.set_aspect('equal')
    ax_l.set_xlabel('X [mm]')
    ax_l.set_ylabel('Y [mm]')
    ax_l.set_xticks([])
    ax_l.set_yticks([])
    ax_l.legend(fontsize=8, loc='upper right')
    ax_l.grid(True, linestyle=':', alpha=0.4)
    title_l = ax_l.set_title('')
    def init_l():
        active_dot.set_data([], [])
        trail_scatter.set_offsets(np.empty((0, 2)))
        return (active_dot, trail_scatter)
    def update_l(frame):
        x, y = xs[frame], ys[frame]
        active_dot.set_data([x], [y])
        trail_length = min(frame + 1, 15)
        trail_start = max(0, frame - trail_length + 1)
        trail_positions = np.column_stack((xs[trail_start:frame + 1], ys[trail_start:frame + 1]))
        trail_scatter.set_offsets(trail_positions)
        trail_scatter.set_array(np.linspace(0.3, 1.0, len(trail_positions)))
        title_l.set_text(f'{scenario_label} - Position {frame} | ({x:.0f}, {y:.0f}) mm')
        return (active_dot, trail_scatter, title_l)
    ani_l = animation.FuncAnimation(fig_l, update_l, frames=n, init_func=init_l, interval=int(1000 / fps), blit=True)
    plt.close(fig_l)
    return HTML(ani_l.to_jshtml())

print('Moving Listener - room layout animation')
make_layout_animation(ml_x, ml_y, speaker_pos, None, 'Moving Listener', fps=3)

print('Moving Person - room layout animation')
make_layout_animation(mp_x, mp_y, speaker_pos, mic_fixed, 'Moving Person', fps=10)

---
## 7. Response Interpolation - Before and After

In [ ]:
rir_a = torch.from_numpy(rirs_ml[0].astype(np.float32))
rir_b = torch.from_numpy(rirs_ml[1].astype(np.float32))
n_total_frames = 40
transition_start = 10
transition_end = 30
max_len = max(len(rir_a), len(rir_b))
import torch.nn.functional as F
rir_a = F.pad(rir_a, (0, max_len - len(rir_a)))
rir_b = F.pad(rir_b, (0, max_len - len(rir_b)))

def frame_to_rir_no_interp(frame_idx):
    if frame_idx < transition_end:
        return rir_a.numpy()
    return rir_b.numpy()

def frame_to_rir_with_interp(frame_idx):
    if frame_idx <= transition_start:
        return rir_a.numpy()
    if frame_idx >= transition_end:
        return rir_b.numpy()
    t = (frame_idx - transition_start) / (transition_end - transition_start)
    interp = interpolate_IRs(t, rir_a.unsqueeze(0), rir_b.unsqueeze(0))
    return interp.squeeze(0).numpy()

def make_interp_animation(frame_to_rir_fn, title, nfft=NFFT, fps=8):
    freqs_local = np.fft.rfftfreq(nfft, d=1.0 / sr)
    m = freqs_local > 50
    all_frames = [frame_to_rir_fn(i) for i in range(n_total_frames)]
    def get_smooth(rir):
        mag = compute_mag_db(rir, nfft=nfft)
        return log_smooth_curve(freqs_local[m], mag[m], window_pts=201)
    all_f = [get_smooth(r)[0] for r in all_frames]
    all_mag = [get_smooth(r)[1] for r in all_frames]
    y_min = min(np.min(m_) for m_ in all_mag) - 3
    y_max = max(np.max(m_) for m_ in all_mag) + 3
    fig_i, ax_i = plt.subplots(figsize=(8, 4))
    line_i, = ax_i.semilogx([], [], linewidth=1.5, color='darkorange')
    f_a, mag_a = get_smooth(rir_a.numpy())
    f_b, mag_b = get_smooth(rir_b.numpy())
    ax_i.semilogx(f_a, mag_a, 'b--', linewidth=0.8, alpha=0.5, label='RIR 0 (start)')
    ax_i.semilogx(f_b, mag_b, 'r--', linewidth=0.8, alpha=0.5, label='RIR 1 (end)')
    ax_i.set_xlim(50, 20000)
    ax_i.set_ylim(y_min, y_max)
    ax_i.set_xlabel('Frequency [Hz]')
    ax_i.set_ylabel('Magnitude [dB]')
    ax_i.legend(fontsize=8)
    ax_i.grid(True, linestyle=':', alpha=0.6)
    title_i = ax_i.set_title('')
    def init_i():
        line_i.set_data([], [])
        return (line_i,)
    def update_i(frame):
        line_i.set_data(all_f[frame], all_mag[frame])
        region = 'before' if frame < transition_start else ('transitioning' if frame < transition_end else 'after')
        title_i.set_text(f'{title} - Frame {frame} [{region}]')
        return (line_i, title_i)
    ani_i = animation.FuncAnimation(fig_i, update_i, frames=n_total_frames, init_func=init_i, interval=int(1000 / fps), blit=True)
    plt.close(fig_i)
    return HTML(ani_i.to_jshtml())

print('Without interpolation - abrupt switch between positions')
make_interp_animation(frame_to_rir_no_interp, 'No interpolation')

print('With interpolation - smooth crossfade between positions')
make_interp_animation(frame_to_rir_with_interp, 'With interpolation')

---
## 8. MedleyDB Input Signals

In [ ]:
medleydb_dir = root / 'data' / 'MedleyDB'
audio_files = sorted(medleydb_dir.glob('**/*.wav')) + sorted(medleydb_dir.glob('**/*.mp3'))
print(f'Found {len(audio_files)} audio files in {medleydb_dir}')
for f in audio_files:
    print(f'  {f.name}')

---
## 9. Target Response Preview

In [ ]:
from utils.common import get_compensation_EQ_params

rir = rirs_ml[0]
roi = (50.0, 20000.0)
eq_comp = get_compensation_EQ_params(rir, sr, ROI=roi, num_sections=7)
target_mag_resp = eq_comp['target_response_db']
target_mag_freqs = eq_comp['freq_axis_smoothed']
rir_smooth_db = eq_comp.get('rir_mag_db_smoothed', None)

fig, ax = plt.subplots(figsize=(9, 4))
if rir_smooth_db is not None:
    ax.semilogx(target_mag_freqs, rir_smooth_db, 'b--', linewidth=1.2, label='RIR (smoothed)')
ax.semilogx(target_mag_freqs, target_mag_resp, 'r-', linewidth=1.4, label='Compensation target')
ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.set_xlim(roi)
ax.set_ylim(-15.0, 5.0)
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Magnitude [dB]')
ax.set_title('Compensation target response (computed from first moving-listener RIR)')
ax.legend()
ax.grid(True, linestyle=':', alpha=0.7)
fig.tight_layout()
plt.show()